# 02b · Edit check only

The last cell of 02_pgd as its own notebook, for platforms where long jobs
must run as a saved version.

**Kaggle:** attach the 01_data output (Add Input → Your Work), set GPU T4 x2
and Internet On, then **Save Version → Save & Run All**. It runs in the
background (you can close the tab; ~1–1.5 h) and the results are kept in that
version's **Output** tab under `soteria/edit_check/`. Interactive sessions get
stopped when idle, which loses everything in `/kaggle/working`.

**Colab:** results go to Google Drive (`MyDrive/soteria`). Runtime → T4 GPU.

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On** (needs a
phone-verified account). Results go to `/kaggle/working/soteria` and are only
kept when you save a version with **Save Version → Save & Run All**; from an
interactive session, download what you need from the Output panel before it ends.
Secrets (`HF_TOKEN`, `GITHUB_TOKEN` for a private fork) go in Add-ons → Secrets.

## Edit check: does a cloak actually spoil a Stable Diffusion edit?

The VAE ratio is only a proxy. This runs SD1.5 img2img (strength 0.5) on 16 of
the images above (12 faces, 4 scenes): the clean photo, two controls and every
PGD run (pgd8, vae_only, rebalanced, pgd16), all with the same prompt and the
same seeded noise. It runs at 256 px (the size the cloaks were made at) and
512 px (SD1.5's native size, the cloak upscaled).

**This cell runs on its own.** Any PGD run whose images are not in
`ROOT/pgd` (e.g. on Kaggle, where the Colab runs are not available) is
computed first for just these 16 images: ~25 min on a T4, then ~30 min of
edits. On Colab with the earlier runs on Drive it goes straight to the edits.

Controls:
- `clean_again`: the clean photo edited a second time. Its score is the
  noise floor of the GPU itself; anything near it means no effect.
- `random8`: random noise at the full 8/255 budget.

Read the results like this:
- `lpips_vs_clean_edit`: how different the edit of the cloaked photo is from
  the edit of the clean one. A cloak works only if this is **clearly above
  `random8`**, not just above 0.
- `id_cos`: ArcFace similarity of the edit to the original person. The
  `clean` row is what the edit keeps by itself; lower than that = identity lost.
- `grid_*.png`: rows are images; columns are the clean input, then the edit of
  clean, clean_again, random8, pgd8, vae_only, rebalanced, pgd16. Look for edits
  that are visibly wrecked (smears, grey patches, wrong content), not just different.

Writes `ROOT/edit_check/`: grid_256.png, grid_512.png, metrics.csv, summary.csv.

In [ ]:
# ---- setup: lets this cell run on its own, on Colab or Kaggle; safe to rerun ----
REPO_URL = "https://github.com/AhnafAbdullah/Soteria-ImageCloaker.git"
BRANCH   = "main"
import importlib, os, subprocess, sys
KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/input")
if not KAGGLE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")                # Colab: results live on Drive
CODE = "/tmp/Project-Soteria" if KAGGLE else "/content/Project-Soteria"
def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print(r.stdout + r.stderr, end=""); r.check_returncode()
def _secret(name):
    try:
        if KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
if os.path.isdir(CODE):
    _git("-C", CODE, "pull", "-q", "--ff-only")
else:
    _token = _secret("GITHUB_TOKEN")             # only needed for a private fork
    _git("clone", "-q", "-b", BRANCH, REPO_URL.replace("https://", f"https://{_token}@") if _token else REPO_URL, CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
_git("log", "-1", "--format=code: %h %s")
import soteria.runtime as rt; importlib.reload(rt)
PLATFORM = rt.platform()
ROOT, DATA = rt.prepare(data=True, gpu=True, requirements=('requirements.txt',))
# ---------------------------------------------------------------------------------

!python -m soteria.eval.edit data_root={DATA} --make-missing --pgd-root {ROOT}/pgd --out {ROOT}/edit_check
import pandas as pd
from IPython.display import Image as Show, display
display(pd.read_csv(f"{ROOT}/edit_check/summary.csv").round(4))
for size in (256, 512):
    print(f"grid {size}px: clean input | edit of clean, clean_again, random8, pgd8, vae_only, rebalanced, pgd16")
    display(Show(f"{ROOT}/edit_check/grid_{size}.png", width=900))